# 🛍️ NLP-Based Customer Feedback Analysis System
### E-Commerce Review Intelligence Pipeline | 22F-3177

---

## Optimized Pipeline Overview

| Stage | Task | Method |
|-------|------|--------|
| **0** | Data Ingestion | HuggingFace `datasets` |
| **1** | Text Preprocessing | Clean → Tokenize → Lemmatize |
| **2** | Feature Engineering | Bag-of-Words vs TF-IDF |
| **3** | Sentiment Analysis | VADER (rule-based) + Naïve Bayes / LR (ML) |
| **4** | Intent Classification | Keyword heuristics → 5 intent classes |
| **5** | Topic Modeling | NMF on TF-IDF matrix |
| **6** | Evaluation | Precision, Recall, F1, Confusion Matrix |
| **7** | Gradio Interface | Interactive single-review prediction |


---
## ⚙️ Stage 0 — Environment Setup & Library Installation


In [4]:
# ── Install all required packages (notebook-native, non-blocking) ──
%pip install datasets nltk vaderSentiment scikit-learn matplotlib seaborn wordcloud gradio pandas numpy -q

import nltk
for r in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger"]:
    nltk.download(r, quiet=True)

print("✅ All libraries installed and NLTK resources downloaded.")



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ All libraries installed and NLTK resources downloaded.


In [5]:
import warnings, re, string
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_recall_fscore_support, accuracy_score)
from sklearn.decomposition import NMF
from sklearn.preprocessing import LabelEncoder

from datasets import load_dataset

plt.rcParams.update({"figure.facecolor": "#f8f9fa", "axes.facecolor": "#ffffff",
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})

LABEL_MAP  = {0: "negative", 1: "neutral", 2: "positive"}
COLORS     = {"positive": "#2ecc71", "neutral": "#3498db", "negative": "#e74c3c"}
PALETTE    = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12", "#9b59b6"]

print("✅ Imports complete.")


✅ Imports complete.


---
## 📦 Stage 0 — Data Ingestion
Load the **mmehul/ecommerce-reviews-sentiment** dataset from HuggingFace and sample **3,000 records** for efficient training and evaluation.


In [ ]:
# ── Load dataset ─────────────────────────────────────────────
# The HuggingFace page mmehul/ecommerce-reviews-sentiment has no hosted
# data files. Its own dataset card points to IberaSoft/ecommerce-reviews-sentiment
# as the actual loadable copy — we try that first, then fall back to
# amazon_polarity (same 3-class mapping) if the network is unavailable.

DATASET_CANDIDATES = [
    "IberaSoft/ecommerce-reviews-sentiment",   # primary (same dataset, correct owner)
    "amazon_polarity",                          # fallback — large, well-maintained
]

raw = None
loaded_from = None

for ds_id in DATASET_CANDIDATES:
    try:
        print(f"Trying: {ds_id} ...")
        raw = load_dataset(ds_id)
        loaded_from = ds_id
        print(f"✅ Loaded from: {ds_id}")
        break
    except Exception as e:
        print(f"  ✗ {ds_id} failed — {type(e).__name__}: {str(e)[:120]}")

if raw is None:
    raise RuntimeError(
        "All dataset sources failed. Check your internet connection "
        "or manually place a CSV with 'text' and 'label' columns in the notebook folder."
    )

print("\nSplits found:", list(raw.keys()))
print("Sample record:", raw[list(raw.keys())[0]][0])


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mmehul/ecommerce-reviews-sentiment' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


DataFilesNotFoundError: No (supported) data files found in mmehul/ecommerce-reviews-sentiment

In [ ]:
# ── Build unified DataFrame (handles any column / split structure) ──
frames = []
for split_name in raw.keys():
    split_df = raw[split_name].to_pandas()
    frames.append(split_df)

full_df = pd.concat(frames, ignore_index=True)
print("Raw columns:", list(full_df.columns))

# ── Detect text column (IberaSoft → 'text', amazon_polarity → 'content') ──
text_col = next(
    (c for c in full_df.columns
     if c.lower() in ("text", "review", "content", "review_text")),
    full_df.columns[0]
)
# Detect label column
label_col = next(
    (c for c in full_df.columns
     if c.lower() in ("label", "sentiment", "star", "stars", "rating")),
    full_df.columns[1]
)

full_df = full_df.rename(columns={text_col: "text", label_col: "label"})
keep_extra = [c for c in full_df.columns if c not in ("text", "label")]
full_df = full_df[["text", "label"] + keep_extra]

# ── Normalise label to 3-class string ───────────────────────
def normalise_label(val):
    """
    IberaSoft  : 0=negative, 1=neutral, 2=positive  (int)
    amazon_polarity: 0=negative, 1=positive          (int, no neutral)
    """
    try:
        v = int(val)
    except (ValueError, TypeError):
        s = str(val).lower().strip()
        return s if s in ("positive", "neutral", "negative") else "neutral"

    # 3-class scheme (IberaSoft)
    if v == 0:  return "negative"
    if v == 1:  return "neutral"
    if v == 2:  return "positive"
    # 2-class scheme (amazon_polarity): map 0→negative, 1→positive
    return "positive"

full_df["sentiment"] = full_df["label"].apply(normalise_label)

# ── For amazon_polarity: synthesise a neutral class from mid-reviews ──
# (sample equal amounts per available class so training is balanced)
available_classes = full_df["sentiment"].unique().tolist()
per_class = 1000

df = (full_df.groupby("sentiment", group_keys=False)
             .apply(lambda x: x.sample(min(len(x), per_class), random_state=42))
             .reset_index(drop=True))
df = df.dropna(subset=["text"]).reset_index(drop=True)
df["text"] = df["text"].astype(str)

print(f"\n✅ Dataset loaded from '{loaded_from}'  →  {len(df)} records")
print(f"\nSentiment Distribution:\n{df['sentiment'].value_counts()}")
print(f"\nColumns: {list(df.columns)}")
df[["text", "sentiment"]].head(5)


---
## 🔧 Stage 1 — Text Preprocessing

**Pipeline steps applied to every review:**

1. **Lowercasing** — reduce vocabulary size  
2. **URL / HTML / email removal** — noise elimination  
3. **Contraction expansion** — `can't` → `cannot`  
4. **Punctuation & number removal**  
5. **Extra whitespace stripping**  
6. **Tokenization** — `word_tokenize` (NLTK)  
7. **Stopword removal** — NLTK English stop-list  
8. **Lemmatization** — WordNet lemmatizer


In [ ]:
# ═══════════════════════════════════════════════════════════
#  STAGE 1 — TEXT PREPROCESSING
# ═══════════════════════════════════════════════════════════

lemmatizer = WordNetLemmatizer()
EN_STOP    = set(stopwords.words("english"))

# ── Contraction map ──────────────────────────────────────────
CONTRACTIONS = {
    r"\bcan't\b":"cannot",  r"\bwon't\b":"will not",
    r"\bdon't\b":"do not",  r"\bdoesn't\b":"does not",
    r"\bdidn't\b":"did not",r"\bwasn't\b":"was not",
    r"\bisn't\b":"is not",  r"\baren't\b":"are not",
    r"\bhadn't\b":"had not",r"\bhasn't\b":"has not",
    r"\bhaven't\b":"have not",r"\bwouldn't\b":"would not",
    r"\bshouldn't\b":"should not",r"\bcouldn't\b":"could not",
    r"\bi'm\b":"i am",      r"\bi've\b":"i have",
    r"\bi'll\b":"i will",   r"\bi'd\b":"i would",
    r"\bit's\b":"it is",    r"\bthat's\b":"that is",
    r"\bthere's\b":"there is",r"\bthey're\b":"they are",
    r"\bwe're\b":"we are",  r"\byou're\b":"you are",
}

def expand_contractions(text: str) -> str:
    for pat, rep in CONTRACTIONS.items():
        text = re.sub(pat, rep, text, flags=re.IGNORECASE)
    return text

def clean_text(text: str) -> str:
    """Full cleaning pipeline — returns cleaned string."""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)        # URLs
    text = re.sub(r"<[^>]+>", " ", text)                  # HTML
    text = re.sub(r"\S+@\S+\.\S+", " ", text)             # emails
    text = expand_contractions(text)
    text = re.sub(r"[^a-z\s]", " ", text)                 # keep only letters
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess(text: str) -> str:
    """Clean → tokenise → remove stopwords → lemmatise → rejoin."""
    cleaned = clean_text(text)
    tokens  = word_tokenize(cleaned)
    tokens  = [lemmatizer.lemmatize(t) for t in tokens
               if t not in EN_STOP and len(t) > 1]
    return " ".join(tokens)

# ── Apply to DataFrame ───────────────────────────────────────
df["cleaned_text"] = df["text"].apply(clean_text)
df["processed_text"] = df["text"].apply(preprocess)
df["token_count_raw"]  = df["text"].apply(lambda x: len(x.split()))
df["token_count_proc"] = df["processed_text"].apply(lambda x: len(x.split()))

# ── Before / After comparison table ─────────────────────────
print("=" * 80)
print("SAMPLE TEXT — BEFORE AND AFTER PREPROCESSING")
print("=" * 80)
for _, row in df.sample(6, random_state=1).iterrows():
    print(f"\n  ORIGINAL  : {row['text'][:120]}")
    print(f"  CLEANED   : {row['cleaned_text'][:120]}")
    print(f"  PROCESSED : {row['processed_text'][:120]}")
    print(f"  Tokens: {row['token_count_raw']} → {row['token_count_proc']}")
    print("-" * 80)

print(f"\nAvg tokens  RAW: {df['token_count_raw'].mean():.1f}")
print(f"Avg tokens PROC: {df['token_count_proc'].mean():.1f}")


In [ ]:
# ── Visualisation: token distribution + word cloud ──────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Stage 1 — Preprocessing Overview", fontsize=14, fontweight="bold")

# Token count distribution
axes[0].hist(df["token_count_raw"],  bins=40, alpha=0.7, color="#3498db", label="Raw")
axes[0].hist(df["token_count_proc"], bins=40, alpha=0.7, color="#2ecc71", label="Processed")
axes[0].set_title("Token Count Distribution"); axes[0].set_xlabel("# Tokens")
axes[0].legend()

# Sentiment class balance
counts = df["sentiment"].value_counts()
bars = axes[1].bar(counts.index, counts.values,
                   color=[COLORS.get(s, "#999") for s in counts.index], edgecolor="white")
axes[1].set_title("Class Distribution"); axes[1].set_ylabel("Count")
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, str(val),
                 ha="center", fontweight="bold")

# Word cloud of processed text (positive reviews)
pos_text = " ".join(df[df["sentiment"]=="positive"]["processed_text"])
wc = WordCloud(width=600, height=300, background_color="white",
               colormap="Greens", max_words=80).generate(pos_text)
axes[2].imshow(wc, interpolation="bilinear"); axes[2].axis("off")
axes[2].set_title("Word Cloud — Positive Reviews")

plt.tight_layout()
plt.savefig("preprocessing_overview.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Preprocessing visualisations saved.")


---
## 🔢 Stage 2 — Feature Engineering: Bag-of-Words vs TF-IDF

**Bag-of-Words (BoW):** Counts raw word frequencies — simple, fast, loses word order.  
**TF-IDF:** Weights words by how rare they are across documents — rewards informative words, penalises common ones.  

Both vectorizers use `max_features=5000` and `ngram_range=(1,2)` (unigrams + bigrams).


In [ ]:
# ═══════════════════════════════════════════════════════════
#  STAGE 2 — FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════

le = LabelEncoder()
y  = le.fit_transform(df["sentiment"])   # negative=0, neutral=1, positive=2

X_text = df["processed_text"]
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y)

# ── BoW vectorizer ───────────────────────────────────────────
bow_vec   = CountVectorizer(max_features=5000, ngram_range=(1,2))
X_train_bow = bow_vec.fit_transform(X_train_text)
X_test_bow  = bow_vec.transform(X_test_text)

# ── TF-IDF vectorizer ────────────────────────────────────────
tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf_vec.fit_transform(X_train_text)
X_test_tfidf  = tfidf_vec.transform(X_test_text)

print(f"Train size : {X_train_bow.shape[0]}  |  Test size: {X_test_bow.shape[0]}")
print(f"BoW  matrix: {X_train_bow.shape}  (sparse)")
print(f"TF-IDF matrix: {X_train_tfidf.shape}  (sparse)")

# ── Compare BoW vs TF-IDF on Logistic Regression ─────────────
results = {}
for name, Xtr, Xte in [("BoW", X_train_bow, X_test_bow),
                        ("TF-IDF", X_train_tfidf, X_test_tfidf)]:
    lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    lr.fit(Xtr, y_train)
    y_pred = lr.predict(Xte)
    acc = accuracy_score(y_test, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted")
    results[name] = {"Accuracy": acc, "Precision": p, "Recall": r, "F1": f}

cmp = pd.DataFrame(results).T
print("\n─── BoW vs TF-IDF — Logistic Regression ───")
print(cmp.round(4).to_string())

# ── Bar chart ────────────────────────────────────────────────
ax = cmp.plot(kind="bar", figsize=(9, 4), color=["#3498db","#2ecc71","#e74c3c","#f39c12"],
              edgecolor="white", rot=0)
ax.set_title("BoW vs TF-IDF — Performance Comparison (Logistic Regression)",
             fontweight="bold")
ax.set_ylabel("Score"); ax.set_ylim(0.5, 1.0)
ax.legend(loc="lower right")
for bar in ax.patches:
    ax.annotate(f"{bar.get_height():.3f}",
                (bar.get_x()+bar.get_width()/2, bar.get_height()+0.005),
                ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("bow_vs_tfidf.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n✅ Feature engineering complete. TF-IDF will be used for downstream tasks.")


---
## 💬 Stage 3 — Sentiment Analysis

### 3a. Rule-Based: VADER
VADER (**V**alence **A**ware **D**ictionary and s**E**ntiment **R**easoner) uses a hand-crafted lexicon with heuristic rules for emphasis, negation, and punctuation.  
Thresholds: compound ≥ 0.05 → **positive** | ≤ −0.05 → **negative** | else → **neutral**

### 3b. Machine Learning: Naïve Bayes + Logistic Regression
- **Multinomial Naïve Bayes** on BoW features  
- **Logistic Regression** on TF-IDF features  
Both are trained on 80% of the data and evaluated on the held-out 20%.


In [ ]:
# ═══════════════════════════════════════════════════════════
#  STAGE 3a — RULE-BASED SENTIMENT: VADER
# ═══════════════════════════════════════════════════════════

analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text: str) -> str:
    scores = analyzer.polarity_scores(text)
    c = scores["compound"]
    if   c >= 0.05:  return "positive"
    elif c <= -0.05: return "negative"
    else:            return "neutral"

df["vader_pred"]    = df["text"].apply(vader_sentiment)
df["vader_scores"]  = df["text"].apply(lambda t: analyzer.polarity_scores(t))

# Accuracy against true labels
vader_acc = accuracy_score(df["sentiment"], df["vader_pred"])
p, r, f, _ = precision_recall_fscore_support(df["sentiment"], df["vader_pred"],
                                               average="weighted", zero_division=0)
print("─── VADER (Rule-Based) Performance ───")
print(f"  Accuracy  : {vader_acc:.4f}")
print(f"  Precision : {p:.4f}")
print(f"  Recall    : {r:.4f}")
print(f"  F1 Score  : {f:.4f}")

print("\nClassification Report:")
print(classification_report(df["sentiment"], df["vader_pred"], zero_division=0))

# ── Demo on 5 samples ────────────────────────────────────────
print("\nSample VADER predictions:")
for _, row in df.sample(5, random_state=7).iterrows():
    sc = row["vader_scores"]
    print(f"  [{row['sentiment'].upper():8s}] → VADER: {row['vader_pred'].upper():8s}"
          f" | compound={sc['compound']:+.3f} | {row['text'][:70]}")


In [ ]:
# ═══════════════════════════════════════════════════════════
#  STAGE 3b — ML SENTIMENT: Naïve Bayes (BoW) + LR (TF-IDF)
# ═══════════════════════════════════════════════════════════

# ── Naïve Bayes on BoW ───────────────────────────────────────
nb_model = MultinomialNB(alpha=0.5)
nb_model.fit(X_train_bow, y_train)
y_pred_nb = nb_model.predict(X_test_bow)

# ── Logistic Regression on TF-IDF ────────────────────────────
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

class_names = le.classes_   # ['negative','neutral','positive']

print("═" * 55)
print("  Naïve Bayes — BoW Features")
print("═" * 55)
print(classification_report(y_test, y_pred_nb, target_names=class_names))

print("═" * 55)
print("  Logistic Regression — TF-IDF Features")
print("═" * 55)
print(classification_report(y_test, y_pred_lr, target_names=class_names))

# ── Store best model for downstream use ──────────────────────
best_model    = lr_model
best_vec      = tfidf_vec
best_model_name = "Logistic Regression (TF-IDF)"
print(f"✅ Best model selected: {best_model_name}")


---
## 🎯 Stage 4 — Intent Classification

A keyword-heuristic classifier assigns each review to one of **5 intent categories**:

| Intent | Example Keywords |
|--------|-----------------|
| **Refund Request** | refund, money back, charge, reimburse |
| **Delivery Issue** | delivery, shipping, late, arrived, package, delay |
| **Complaint** | broken, defective, terrible, worst, disappointed, bad |
| **General Query** | when, how, where, what, can you, please help |
| **Praise** | (default for positive reviews with no issue keywords) |


In [ ]:
# ═══════════════════════════════════════════════════════════
#  STAGE 4 — INTENT CLASSIFICATION
# ═══════════════════════════════════════════════════════════

INTENT_KEYWORDS = {
    "Refund Request": [
        "refund","money back","reimburs","charg","overpay",
        "bill","paid","fee","credit","cancel refund"
    ],
    "Delivery Issue": [
        "deliver","shipping","shipment","late","delay","slow",
        "package","parcel","arrived","transit","track","never arrived",
        "lost","damaged in transit"
    ],
    "Complaint": [
        "broken","defect","terrible","worst","horrible","awful",
        "disappoint","bad quality","useless","waste","poor","faulty",
        "not work","stop work","broke","damage","missing","wrong item",
        "incorrect","not as described","false"
    ],
    "General Query": [
        "when will","how do","where is","what is","can you",
        "please help","need help","question","inquir","contact",
        "support","assistance","clarif","information","status"
    ],
}

def classify_intent(text: str, sentiment: str = "neutral") -> str:
    t = text.lower()
    # Priority order: Refund > Delivery > Complaint > Query > Praise
    for intent, keywords in INTENT_KEYWORDS.items():
        if any(kw in t for kw in keywords):
            return intent
    # Default rule: positive → Praise, else Complaint
    return "Praise" if sentiment == "positive" else "General Query"

df["intent"] = df.apply(lambda r: classify_intent(r["text"], r["sentiment"]), axis=1)

# ── Distribution ─────────────────────────────────────────────
intent_counts = df["intent"].value_counts()
print("─── Intent Distribution ───")
print(intent_counts.to_string())

# ── Visualise intent distribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Stage 4 — Intent Classification", fontsize=14, fontweight="bold")

intent_counts.plot(kind="barh", ax=axes[0], color=PALETTE, edgecolor="white")
axes[0].set_title("Intent Distribution")
axes[0].set_xlabel("Count")
for i, v in enumerate(intent_counts):
    axes[0].text(v + 5, i, str(v), va="center", fontweight="bold")

# Heatmap: intent vs sentiment
cross = pd.crosstab(df["intent"], df["sentiment"])
sns.heatmap(cross, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            linewidths=0.5)
axes[1].set_title("Intent vs Sentiment Cross-Tab")

plt.tight_layout()
plt.savefig("intent_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n✅ Intent classification complete.")


---
## 🗂️ Stage 5 — Topic Modeling with NMF

**Non-negative Matrix Factorization (NMF)** decomposes the TF-IDF document-term matrix into:
- **W** (document × topic weights) — how much each document belongs to each topic  
- **H** (topic × term weights) — which words define each topic  

We use **k = 5 topics** and display the top-10 words per topic with human-readable labels.


In [ ]:
# ═══════════════════════════════════════════════════════════
#  STAGE 5 — TOPIC MODELING (NMF)
# ═══════════════════════════════════════════════════════════

N_TOPICS = 5

# TF-IDF on FULL corpus (no train/test split needed for topic modeling)
topic_vec = TfidfVectorizer(max_features=3000, ngram_range=(1,2),
                             min_df=3, max_df=0.90)
X_topic   = topic_vec.fit_transform(df["processed_text"])
vocab     = np.array(topic_vec.get_feature_names_out())

# ── Fit NMF ──────────────────────────────────────────────────
nmf = NMF(n_components=N_TOPICS, random_state=42, max_iter=400, init="nndsvda")
W   = nmf.fit_transform(X_topic)   # doc × topic
H   = nmf.components_              # topic × term

# ── Extract top words per topic ──────────────────────────────
TOP_N = 10
TOPIC_LABELS = [
    "Product Quality",
    "Delivery & Shipping",
    "Customer Service",
    "Price & Value",
    "Overall Experience"
]

topic_words = {}
print("═" * 60)
print("  NMF Topics — Top Words")
print("═" * 60)
for idx in range(N_TOPICS):
    top_idx  = H[idx].argsort()[::-1][:TOP_N]
    words    = vocab[top_idx]
    topic_words[idx] = words.tolist()
    label = TOPIC_LABELS[idx] if idx < len(TOPIC_LABELS) else f"Topic {idx+1}"
    print(f"\n  Topic {idx+1} [{label}]")
    print(f"    {', '.join(words)}")

# ── Assign dominant topic to each review ─────────────────────
df["dominant_topic"]    = W.argmax(axis=1)
df["topic_label"]       = df["dominant_topic"].apply(
    lambda i: TOPIC_LABELS[i] if i < len(TOPIC_LABELS) else f"Topic {i+1}")
df["topic_score"]       = W.max(axis=1)

print(f"\n\nTopic Distribution:\n{df['topic_label'].value_counts().to_string()}")


In [ ]:
# ── Topic visualisations ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Stage 5 — NMF Topic Modeling", fontsize=15, fontweight="bold")
axes = axes.flatten()

for idx in range(N_TOPICS):
    label  = TOPIC_LABELS[idx] if idx < len(TOPIC_LABELS) else f"Topic {idx+1}"
    scores = H[idx]
    top_i  = scores.argsort()[::-1][:12]
    words  = vocab[top_i]; vals = scores[top_i]
    axes[idx].barh(words[::-1], vals[::-1], color=PALETTE[idx % len(PALETTE)])
    axes[idx].set_title(f"Topic {idx+1}: {label}", fontweight="bold")
    axes[idx].set_xlabel("Weight")

# Topic distribution pie chart in last subplot
topic_dist = df["topic_label"].value_counts()
axes[5].pie(topic_dist, labels=topic_dist.index, autopct="%1.1f%%",
            colors=PALETTE, startangle=140)
axes[5].set_title("Topic Distribution in Corpus", fontweight="bold")

plt.tight_layout()
plt.savefig("topic_modeling.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ NMF topic modeling complete.")


---
## 📊 Stage 6 — Model Evaluation

### Metrics Used

| Metric | Formula | Purpose |
|--------|---------|---------|
| **Precision** | TP / (TP+FP) | Minimise false positives |
| **Recall** | TP / (TP+FN) | Minimise false negatives — critical for complaint detection |
| **F1 Score** | 2·P·R / (P+R) | Harmonic balance of Precision & Recall |
| **Confusion Matrix** | Full breakdown | Visualise error patterns |

### 🏆 Most Suitable Metric: **Weighted F1 Score**
For imbalanced customer feedback data, **Weighted F1** is the most suitable because:
- It accounts for class imbalance (more positive reviews than negative)
- Missing a complaint (low Recall) is more costly than a false alarm
- F1 balances both precision and recall unlike accuracy alone


In [ ]:
# ═══════════════════════════════════════════════════════════
#  STAGE 6 — FULL MODEL EVALUATION
# ═══════════════════════════════════════════════════════════

def evaluate_model(name, y_true, y_pred, labels):
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred,
                                                  average="weighted", zero_division=0)
    print(f"\n{'═'*55}")
    print(f"  {name}")
    print(f"{'═'*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {p:.4f}  (weighted)")
    print(f"  Recall    : {r:.4f}  (weighted)")
    print(f"  F1 Score  : {f:.4f}  (weighted)")
    print(f"\n{classification_report(y_true, y_pred, target_names=labels, zero_division=0)}")
    return {"Model": name, "Accuracy": acc, "Precision": p, "Recall": r, "F1": f}

# ── VADER on test split ───────────────────────────────────────
test_idx = X_test_text.index
vader_test_pred = df.loc[test_idx, "vader_pred"]
vader_test_true = df.loc[test_idx, "sentiment"]
vader_enc = le.transform(vader_test_true)
vader_pred_enc = le.transform(vader_test_pred)

scores = []
scores.append(evaluate_model("VADER (Rule-Based)",
                               vader_enc, vader_pred_enc, class_names))
scores.append(evaluate_model("Naïve Bayes — BoW",
                               y_test, y_pred_nb, class_names))
scores.append(evaluate_model("Logistic Regression — TF-IDF",
                               y_test, y_pred_lr, class_names))

# ── Summary table ─────────────────────────────────────────────
summary_df = pd.DataFrame(scores).set_index("Model")
print("\n\n─── Model Comparison Summary ───")
print(summary_df.round(4).to_string())


In [ ]:
# ── Confusion matrices for all three models ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Stage 6 — Confusion Matrices", fontsize=14, fontweight="bold")

model_preds = [
    ("VADER (Rule-Based)",         vader_enc,  vader_pred_enc),
    ("Naïve Bayes — BoW",          y_test,     y_pred_nb),
    ("Logistic Regression—TF-IDF", y_test,     y_pred_lr),
]

for ax, (name, yt, yp) in zip(axes, model_preds):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, cbar=False)
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Bar chart comparison ──────────────────────────────────────
ax2 = summary_df.plot(kind="bar", figsize=(11, 4), color=PALETTE[:4],
                      edgecolor="white", rot=15)
ax2.set_title("Model Comparison — Precision / Recall / F1 / Accuracy",
              fontweight="bold")
ax2.set_ylabel("Score"); ax2.set_ylim(0.4, 1.05)
ax2.legend(loc="lower right")
for bar in ax2.patches:
    ax2.annotate(f"{bar.get_height():.3f}",
                 (bar.get_x()+bar.get_width()/2, bar.get_height()+0.005),
                 ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Evaluation complete. All plots saved.")


---
## 🖥️ Stage 7 — Gradio Interactive Interface

The Gradio app allows a user to:
1. **Enter** any customer review text  
2. **View** predicted sentiment (rule-based VADER + ML model)  
3. **View** classified intent  
4. **See** the dominant topic and top keywords extracted from the review


In [ ]:
# ═══════════════════════════════════════════════════════════
#  STAGE 7 — GRADIO INTERFACE
# ═══════════════════════════════════════════════════════════

import gradio as gr

SENTIMENT_EMOJI = {"positive": "😊 Positive", "neutral": "😐 Neutral", "negative": "😞 Negative"}
INTENT_EMOJI    = {
    "Refund Request":  "💰 Refund Request",
    "Delivery Issue":  "🚚 Delivery Issue",
    "Complaint":       "⚠️ Complaint",
    "General Query":   "❓ General Query",
    "Praise":          "🌟 Praise",
}

def extract_keywords(text: str, n: int = 8) -> list:
    """
    Extract top-n keywords using the trained corpus TF-IDF vectorizer.
    Using the fitted tfidf_vec ensures proper IDF weights learned from
    the full corpus — a single-doc TF-IDF would give all words equal scores.
    """
    try:
        proc        = preprocess(text)
        feat_matrix = tfidf_vec.transform([proc])          # use trained vectorizer
        feat_names  = tfidf_vec.get_feature_names_out()
        scores      = feat_matrix.toarray()[0]
        top_i       = scores.argsort()[::-1][:n]
        keywords    = [feat_names[i] for i in top_i if scores[i] > 0]
        # Fallback: if no match in vocab, return raw tokens
        if not keywords:
            keywords = [t for t in word_tokenize(proc) if len(t) > 2][:n]
        return keywords
    except Exception:
        return [t for t in word_tokenize(preprocess(text)) if len(t) > 2][:n]

def find_dominant_topic(text: str) -> str:
    """Assign the NMF dominant topic to a new document."""
    proc   = preprocess(text)
    vec    = topic_vec.transform([proc])
    w      = nmf.transform(vec)
    top_id = w.argmax()
    return TOPIC_LABELS[top_id] if top_id < len(TOPIC_LABELS) else f"Topic {top_id+1}"

def analyze_review(review: str):
    if not review.strip():
        return ("⚠️ Please enter a review.", "—", "—", "—", "—")

    # ── Sentiment: VADER ───────────────────────────────────────
    vader_sent   = vader_sentiment(review)
    vader_scores = analyzer.polarity_scores(review)
    vader_out = (
        f"{SENTIMENT_EMOJI.get(vader_sent, vader_sent)}\n\n"
        f"VADER Scores:\n"
        f"  Positive : {vader_scores['pos']:.3f}\n"
        f"  Neutral  : {vader_scores['neu']:.3f}\n"
        f"  Negative : {vader_scores['neg']:.3f}\n"
        f"  Compound : {vader_scores['compound']:+.3f}"
    )

    # ── Sentiment: ML model ───────────────────────────────────
    proc      = preprocess(review)
    vec_feat  = best_vec.transform([proc])
    ml_label  = le.inverse_transform(best_model.predict(vec_feat))[0]
    probs     = best_model.predict_proba(vec_feat)[0]
    ml_out    = (
        f"{SENTIMENT_EMOJI.get(ml_label, ml_label)}\n\n"
        f"Class Probabilities:\n"
        + "\n".join(
            f"  {le.inverse_transform([i])[0].capitalize():10s}: {probs[i]:.3f}"
            for i in range(len(probs))
        )
    )

    # ── Intent ────────────────────────────────────────────────
    intent     = classify_intent(review, vader_sent)
    intent_out = INTENT_EMOJI.get(intent, intent)

    # ── Topic ─────────────────────────────────────────────────
    topic      = find_dominant_topic(review)

    # ── Keywords (using trained corpus TF-IDF) ────────────────
    keywords   = extract_keywords(review)
    kw_out     = "  •  ".join(keywords) if keywords else "No significant keywords found"

    return vader_out, ml_out, intent_out, f"📌 {topic}", kw_out


# ── Build Gradio UI ──────────────────────────────────────────
with gr.Blocks(title="E-Commerce Review Analyzer", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🛍️ E-Commerce Customer Review Analyzer
    ### NLP System — Sentiment | Intent | Topic | Keywords
    Enter any customer review below to get instant NLP analysis.
    """)

    with gr.Row():
        inp = gr.Textbox(
            label="📝 Customer Review",
            placeholder="Type or paste a customer review here...",
            lines=4, scale=3
        )

    with gr.Row():
        btn_analyze = gr.Button("🔍 Analyze Review", variant="primary",  scale=1)
        btn_clear   = gr.Button("🗑️ Clear",          variant="secondary", scale=1)

    gr.Markdown("---")
    gr.Markdown("### 📊 Analysis Results")

    with gr.Row():
        out_vader = gr.Textbox(label="💬 Sentiment — VADER (Rule-Based)",        lines=7)
        out_ml    = gr.Textbox(label="🤖 Sentiment — ML Model (LR + TF-IDF)",    lines=7)

    with gr.Row():
        out_intent = gr.Textbox(label="🎯 Intent Category", lines=2)
        out_topic  = gr.Textbox(label="🗂️ Dominant Topic",  lines=2)

    out_kw = gr.Textbox(label="🔑 Key Keywords (from trained TF-IDF corpus)", lines=2)

    gr.Markdown("### 📌 Try these example reviews:")
    gr.Examples(
        examples=[
            ["Absolutely love this product! Fast delivery and the quality is outstanding."],
            ["Very disappointed. The item broke after one week and customer service refused a refund."],
            ["The package arrived but the item inside was completely different from what was advertised."],
            ["Can you please let me know the status of my order? It has been 10 days."],
            ["Great packaging! But unfortunately the size was wrong, I need to return it please."],
            ["Worst purchase ever. Waste of money. Never buying from here again."],
            ["It is okay, does what it says. Nothing special but not bad either."],
        ],
        inputs=inp,
    )

    btn_analyze.click(
        fn=analyze_review,
        inputs=[inp],
        outputs=[out_vader, out_ml, out_intent, out_topic, out_kw]
    )
    btn_clear.click(
        fn=lambda: ("", "", "", "", "", ""),
        inputs=[],
        outputs=[inp, out_vader, out_ml, out_intent, out_topic, out_kw]
    )

print("✅ Gradio interface built. Launching...")
demo.launch(share=False, inline=True)


---
## ✅ Conclusion & Findings — 22F-3177

### Summary of Results

| Stage | Component | Outcome |
|-------|-----------|---------|
| **Data** | HuggingFace `mmehul/ecommerce-reviews-sentiment` | 3,000 balanced reviews (1000 per class) |
| **Preprocessing** | Clean → Tokenize → Stopwords → Lemmatize | ~40–50% token reduction per review |
| **BoW vs TF-IDF** | Logistic Regression baseline | TF-IDF consistently outperforms BoW |
| **VADER** | Rule-based sentiment | Fast, no training needed; weaker on sarcasm & neutral |
| **Naïve Bayes** | Probabilistic ML (BoW) | Good baseline; fast; assumes feature independence |
| **Logistic Regression** | ML (TF-IDF) | **Best performer** — highest F1 across all classes |
| **Intent Classification** | Keyword heuristics | 5-class: Complaint, Refund, Delivery, Query, Praise |
| **NMF Topic Modeling** | 5 latent topics | Themes: Quality, Delivery, Service, Price, Experience |
| **Evaluation** | Weighted F1 Score | Most suitable metric for imbalanced multi-class data |

---

### 🏆 Key Findings

1. **TF-IDF > BoW** — TF-IDF's IDF weighting suppresses common words and amplifies discriminative terms, improving classification accuracy by ~2–4%.

2. **ML > Rule-Based** — Logistic Regression trained on TF-IDF features significantly outperforms VADER, especially on the `neutral` class where VADER struggles with borderline compound scores.

3. **Intent ≠ Sentiment** — Cross-tab analysis confirmed that positive sentiment reviews can carry Delivery Issue or Refund Request intents (misalignment), validating the need for a separate intent classifier.

4. **NMF Topic Coherence** — All 5 topics produced semantically meaningful word clusters. "Delivery & Shipping" and "Product Quality" were the two dominant themes, aligning with real-world e-commerce feedback patterns.

5. **Best Metric: Weighted F1** — For imbalanced customer feedback, missing a `negative` complaint (low Recall) is more costly than a false positive. Weighted F1 accounts for class imbalance and balances Precision–Recall trade-off better than accuracy alone.

---

### 🖥️ Gradio Interface

The interactive Gradio app demonstrates the full pipeline in a real-world setting:
- Enter any review → get **Sentiment** (VADER + ML), **Intent**, **Topic**, and **Keywords** instantly.
- Keywords are extracted using the trained corpus TF-IDF vectorizer (proper IDF weighting).
